# 05 - Benchmark Comparison Dashboard

**Last Updated:** [Auto-generated]
**Purpose:** Analyze and visualize benchmark results from multiple test runs across different suites and models.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any
import warnings

warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [ ]:
# Configuration
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'scripts':
    PROJECT_ROOT = PROJECT_ROOT.parent
    
RESULTS_DIR = PROJECT_ROOT / 'data' / 'benchmark' / 'results' / 'runs'

print(f"Project root: {PROJECT_ROOT}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Results directory exists: {RESULTS_DIR.exists()}")

In [ ]:
def load_all_runs(results_dir: Path) -> List[Dict[str, Any]]:
    """
    Scan results directory for results.json files and load them.
    
    Args:
        results_dir: Path to benchmark results directory
        
    Returns:
        List of dictionaries containing run results
    """
    runs = []
    
    if not results_dir.exists():
        print(f"Warning: Results directory does not exist: {results_dir}")
        return runs
    
    for results_file in results_dir.glob('*/results.json'):
        try:
            with open(results_file, 'r') as f:
                run_data = json.load(f)
                run_data['_path'] = str(results_file.parent)
                runs.append(run_data)
                print(f"Loaded: {results_file.parent.name}")
        except Exception as e:
            print(f"Error loading {results_file}: {e}")
    
    print(f"\nTotal runs loaded: {len(runs)}")
    return runs

# Load all runs
runs = load_all_runs(RESULTS_DIR)

In [ ]:
def runs_to_dataframe(runs: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Flatten aggregate metrics from runs into a DataFrame.
    
    Args:
        runs: List of run dictionaries
        
    Returns:
        DataFrame with flattened metrics
    """
    records = []
    
    for run in runs:
        record = {
            'run_name': run.get('run_name', 'Unknown'),
            'timestamp': run.get('timestamp', ''),
            'suite': run.get('suite', ''),
        }
        
        # Flatten aggregate metrics
        aggregate = run.get('aggregate', {})
        for key, value in aggregate.items():
            if isinstance(value, (int, float)):
                record[f'agg_{key}'] = value
        
        records.append(record)
    
    df = pd.DataFrame(records)
    
    if len(df) == 0:
        print("Warning: No data to create DataFrame")
        return df
    
    # Reorder columns
    cols = [c for c in df.columns if c.startswith('agg_')]
    cols = ['run_name', 'timestamp', 'suite'] + cols
    df = df[[c for c in cols if c in df.columns]]
    
    return df

# Create DataFrame
df_runs = runs_to_dataframe(runs)
print(f"\nDataFrame shape: {df_runs.shape}")
print("\nFirst few rows:")
df_runs.head()

In [ ]:
# Display summary statistics
if len(df_runs) > 0:
    print("Summary Statistics:")
    print("=" * 60)
    for suite in df_runs['suite'].unique():
        if pd.isna(suite):
            continue
        suite_df = df_runs[df_runs['suite'] == suite]
        print(f"\nSuite: {suite}")
        print(f"Number of runs: {len(suite_df)}")
        
        # Show numeric columns
        numeric_cols = suite_df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            print(suite_df[numeric_cols].describe().round(3))
else:
    print("No benchmark results available to analyze.")

In [ ]:
# Bar chart: accuracy per suite
if len(df_runs) > 0:
    accuracy_cols = [c for c in df_runs.columns if 'accuracy' in c.lower()]
    
    if accuracy_cols:
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Group by suite and calculate mean of first accuracy metric
        acc_col = accuracy_cols[0]
        grouped = df_runs.groupby('suite')[acc_col].mean()
        
        grouped.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
        ax.set_title(f'Average {acc_col} by Suite', fontsize=14, fontweight='bold')
        ax.set_xlabel('Suite')
        ax.set_ylabel(acc_col)
        ax.set_ylim([0, 1])
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print("No accuracy metrics found in results")
else:
    print("No data available for plotting")

In [ ]:
# Grouped bar chart: model comparison within vlm_extraction suite
if len(df_runs) > 0 and 'agg_accuracy' in df_runs.columns:
    vlm_df = df_runs[df_runs['suite'] == 'vlm_extraction'].copy() if 'suite' in df_runs.columns else df_runs
    
    if len(vlm_df) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Create grouped bar chart
        metrics = [c for c in vlm_df.columns if c.startswith('agg_')]
        x = np.arange(len(vlm_df))
        width = 0.25
        
        for i, metric in enumerate(metrics[:3]):  # Show top 3 metrics
            values = vlm_df[metric].values
            ax.bar(x + i*width, values, width, label=metric.replace('agg_', ''), alpha=0.8)
        
        ax.set_xlabel('Run')
        ax.set_ylabel('Score')
        ax.set_title('Model Comparison - VLM Extraction Suite', fontsize=14, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels(vlm_df['run_name'].values, rotation=45, ha='right')
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No vlm_extraction suite data available")
else:
    print("Insufficient data for grouped bar chart")

In [ ]:
# Radar chart: multi-metric comparison across suites
if len(df_runs) > 0 and len(df_runs) >= 2:
    from math import pi
    
    # Prepare data for radar chart
    metrics = [c for c in df_runs.columns if c.startswith('agg_')][:5]  # Top 5 metrics
    
    if metrics:
        # Use first run if available
        run_data = df_runs.iloc[0]
        values = [run_data[m] for m in metrics]
        values += values[:1]  # Complete the circle
        
        # Create radar chart
        angles = [n / float(len(metrics)) * 2 * pi for n in range(len(metrics))]
        angles += angles[:1]
        
        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
        ax.plot(angles, values, 'o-', linewidth=2, label=run_data['run_name'])
        ax.fill(angles, values, alpha=0.25)
        
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels([m.replace('agg_', '') for m in metrics], size=10)
        ax.set_ylim(0, 1)
        ax.set_title('Multi-Metric Comparison', fontsize=14, fontweight='bold', pad=20)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        ax.grid(True)
        
        plt.tight_layout()
        plt.show()
    else:
        print("No metrics available for radar chart")
else:
    print("Insufficient data for radar chart")

In [ ]:
def to_latex_table(df: pd.DataFrame, caption: str = "Benchmark Results") -> str:
    """
    Export DataFrame to LaTeX table format.
    
    Args:
        df: DataFrame to export
        caption: Table caption
        
    Returns:
        LaTeX formatted string
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df_display = df[['run_name', 'suite'] + list(numeric_cols)].copy()
    
    # Round numeric columns to 3 decimals
    for col in numeric_cols:
        if col in df_display.columns:
            df_display[col] = df_display[col].round(3)
    
    latex = df_display.to_latex(index=False)
    latex = f"% {caption}\n" + latex
    
    return latex

# Generate LaTeX table
if len(df_runs) > 0:
    latex_table = to_latex_table(df_runs, "Benchmark Results Summary")
    print("LaTeX Table (first 500 chars):")
    print(latex_table[:500])
else:
    print("No data available for LaTeX table")

## Summary

This dashboard provides a comprehensive analysis of benchmark results from the Geo-SLM Chart Analysis project. Key findings:

- **Data Sources:** Loads all results.json files from the benchmark runs directory
- **Metrics Tracked:** Accuracy, precision, recall, F1-score per suite
- **Visualizations:** Bar charts, grouped comparisons, and radar plots for multi-metric analysis
- **Export Options:** LaTeX table format for inclusion in reports

### Next Steps
1. Review error patterns in the Error Analysis notebook (06_error_analysis.ipynb)
2. Test specific models with the Interactive Demo (07_demo_interactive.ipynb)
3. Iterate on model improvements based on findings